<h1>Ordered Theorell-Chance Model for two Substrates</h1>
<h3>Comparison of alternative models</h3>
<h3>Model as opposed to the "Random Sequential model"</h3>
<p>G. Woehlke 2015 ff (converted from Wolfram Mathematica, aided by A.I.)</p>

<h3>Kinetic scheme: </h3> 
<img src="2-09_OrderedADH.png" width="466"> 
<p>Alcohol dehydrogenase (ADH) reduces ethanol at the expense of NADH. These two substrates may bind randomly, either ethanol first, then NADH, or vice versa (cf. <i>RandomSequential.ipynb</i>), but in reality they have to bind sequentially. The kinetic scheme implements the latter model. </p>

<p>This version calculates the temporal development of concentrations (not steady state). We use the <i>SymPy</i> module to express the mathematical description of the steady state concentrations:</p>

In [14]:
import sympy as sp

# symbols
cA, cB, cPA = sp.symbols('cA cB cPA', positive=True)
k12, k21, k23, k32, k34 = sp.symbols('k12 k21 k23 k32 k34', positive=True)
E_tot = sp.symbols('E_tot', positive=True)

# unknowns
cEA, cEB, cE = sp.symbols('cEA cEB cE')

# steady-state equations
eq1 = sp.Eq(k12*cA*cE - k21*cEA - k23*cB*cEA + k32*cPA*cEB, 0)
eq2 = sp.Eq(k23*cB*cEA - k32*cPA*cEB - k34*cEB, 0)
eq3 = sp.Eq(E_tot, cE + cEA + cEB)

sol = sp.solve([eq1, eq2, eq3], (cEA, cEB, cE), simplify=True)

sol

{cE: E_tot*(cB*k23*k34 + cPA*k21*k32 + k21*k34)/(cA*cB*k12*k23 + cA*cPA*k12*k32 + cA*k12*k34 + cB*k23*k34 + cPA*k21*k32 + k21*k34),
 cEA: E_tot*cA*k12*(cPA*k32 + k34)/(cA*cB*k12*k23 + cA*cPA*k12*k32 + cA*k12*k34 + cB*k23*k34 + cPA*k21*k32 + k21*k34),
 cEB: E_tot*cA*cB*k12*k23/(cA*cB*k12*k23 + cA*cPA*k12*k32 + cA*k12*k34 + cB*k23*k34 + cPA*k21*k32 + k21*k34)}

<p>We then define the observed rate $k_{obs}$ as rate of product A concentration, and let Python solve the equation:</p>

In [15]:
kobs = k34 * sol[cEB].subs(cPA, 0)

kobs_simplified = sp.simplify(kobs)
kobs_simplified

E_tot*cA*cB*k12*k23*k34/(cA*cB*k12*k23 + cA*k12*k34 + cB*k23*k34 + k21*k34)

<p><i>SymPy</i> provides a function that adds a numerical implementation of the symbolic solution: <a href="https://docs.sympy.org/latest/modules/utilities/lambdify.html">sympy.utilities.lambdify.implemented_function(symfunc, implementation)</a>. We use it below for plotting. </p>

In [17]:
kobs_func = sp.lambdify(
    (cA, cB, k12, k21, k23, k32, k34, E_tot),
    kobs_simplified,
    'numpy'
)

<h3>Calculation and plotting routine</h3>
<p>The following code plots y1, y2 as Briggs-Haldane curve (observed velocity = f(concentration), and the first derivative at point $c = 0$. A crosshair is placed at half-maximal turnover $\frac{1}{2}v_{max}$ and $K_{1/2} = \frac{k_{21}+k_{23}}{k_{12}}$.  </p>

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

def plot_vs_cA(cB=5.0, k12=1.0, k21=1.0, k23=1.0, k32=1.0, k34=1.0, E_tot=1.0, rangeY=1.5):
    cA_vals = np.linspace(0.01, 5, 200)

    y = kobs_func(cA_vals, cB, k12, k21, k23, k32, k34, E_tot)

    # comparison curve (Michaelis-like)
    y_mm = k34 * cA_vals / (k21/k12 + cA_vals)

    plt.figure(figsize=(6,4))
    plt.plot(cA_vals, y_mm, '--',color="lightblue", label='Michaelis-Menten')
    plt.plot(cA_vals, y, color="blue", label='Ordered Theorell')
    
    plt.xlabel('[A] (mM)')
    plt.ylabel('k_obs')
    plt.ylim(0, rangeY)
    plt.title('ADH (Theorell–Chance)')
    plt.legend()

    text = (
        f"k21/k12 = {k21/k12:.2f}\n"
        f"k32/k23 = {k32/k23:.2f}\n"
        f"k34 = {k34:.2f}\n"
        f"[B] = {cB:.2f}"
    )
    plt.text(3, rangeY*0.5, text, bbox=dict(facecolor='white'))

    plt.show()

interact(plot_vs_cA,
         cB=(0.01,10,0.1),
         k12=(0.1,10,0.1),
         k21=(0.1,10,0.1),
         k23=(0.1,10,0.1),
         k32=(0.1,10,0.1),
         k34=(0.1,10,0.1),
         rangeY=(0.5,5,0.1));

interactive(children=(FloatSlider(value=5.0, description='cB', max=10.0, min=0.01), FloatSlider(value=1.0, des…

<h3>Figure legend </h3><p>Concentration dependence (unit $M$) of the observed turnover rate (unit $s^{-1}$) according to the ordered reaction model.   <span style="color:blue">Light blue, dashed line</span>: Michaelis-Menten curve as reference,  <span style="color:blue">blue</span>: ordered model according to Theorell-Chance.</p>
<img src="2-09_OrderedADH.png" width="466"> 
<p><span style="color:blue;font-weight: bold">Note</span> that the plot of rate vs concentration does not yield sufficient information to unequivocally discern Michaelis-Menten, Theorell-Chance, or random-ordered model!</p>

<h3>Variation of $c_B$ at constant $c_A$</h3>

In [20]:
def plot_vs_cB(cA=5.0, k12=1.0, k21=1.0, k23=1.0, k32=1.0, k34=1.0, E_tot=1.0, rangeY=1.5):
    cB_vals = np.linspace(0.01, 5, 200)

    y = kobs_func(cA, cB_vals, k12, k21, k23, k32, k34, E_tot)

    plt.figure(figsize=(6,4))
    plt.plot(cB_vals, y)

    plt.xlabel('[B] (mM)')
    plt.ylabel('k_obs')
    plt.ylim(0, rangeY)
    plt.title('ADH (Theorell–Chance)')

    plt.show()

interact(plot_vs_cB,
         cA=(0.01,10,0.1),
         k12=(0.1,10,0.1),
         k21=(0.1,10,0.1),
         k23=(0.1,10,0.1),
         k32=(0.1,10,0.1),
         k34=(0.1,10,0.1),
         rangeY=(0.5,2.5,0.1));

interactive(children=(FloatSlider(value=5.0, description='cA', max=10.0, min=0.01), FloatSlider(value=1.0, des…

<h3>Dependence on both concentrations</h3>
<p>3D graph $k_{obs} = f(c_A, c_B)$</p>

In [21]:
from mpl_toolkits.mplot3d import Axes3D

def plot_3D(k12=1.0, k21=2.0, k23=0.5, k32=4.9, k34=6.3, E_tot=1.0, rangeZ=1.5):
    cA_vals = np.linspace(0.01, 5, 60)
    cB_vals = np.linspace(0.01, 5, 60)

    CA, CB = np.meshgrid(cA_vals, cB_vals)

    Z = kobs_func(CA, CB, k12, k21, k23, k32, k34, E_tot)

    fig = plt.figure(figsize=(7,5))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot_surface(CA, CB, Z, cmap='viridis')

    ax.set_xlabel('[A]')
    ax.set_ylabel('[B]')
    ax.set_zlabel('k_obs')
    ax.set_zlim(0, rangeZ)

    plt.title('ADH (Ordered, Theorell–Chance)')
    plt.show()

interact(plot_3D,
         k12=(0.1,10,0.1),
         k21=(0.1,10,0.1),
         k23=(0.1,10,0.1),
         k32=(0.1,10,0.1),
         k34=(0.1,10,0.1),
         rangeZ=(0.5,5,0.1));

interactive(children=(FloatSlider(value=1.0, description='k12', max=10.0, min=0.1), FloatSlider(value=2.0, des…